In [11]:
"This IPYNB file is only used as a test, and a tool for deploying the model. The real Code goes in the same file, but "
"with a .py extension rather than a .ipynb"

'with a .py extension rather than a .ipynb'

In [13]:
from pathlib import Path
import pandas as pd

# Try common working directories (notebook folder and workspace root).
candidate_paths = [
    Path("../../../../SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
    Path("backend/SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate usgs_data_USGS-01646500.csv")
print(f"Using CSV file at: {csv_path}")

df = pd.read_csv(csv_path)
df

Using CSV file at: ..\..\..\..\SOURCES_AND_DATASHEETS\usgs_data_USGS-01646500.csv


,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,snowfall,snow_depth,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,temperature_2m,wind_speed_10m,vapour_pressure_deficit,precip_3hr,precip_24hr,precip_72hr
0,2025-07-09 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,23.75,5.920878,0.222110,NaN,NaN,NaN
1,2025-07-09 05:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,23.10,8.328770,0.100887,NaN,NaN,NaN
2,2025-07-09 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,22.70,7.412853,0.114930,NaN,NaN,NaN
3,2025-07-09 07:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,22.60,7.907718,0.114322,NaN,NaN,NaN
4,2025-07-09 08:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,22.60,6.854196,0.114322,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104127,2026-07-09 23:50:00+00:00,3.4,5290.0,5.6,8.9,343.0,29.2,13.2,0.0,0.0,0.0,0.0,NaN,NaN,24.70,2.747581,0.127692,78.700002,402.000001,626.200002
104128,2026-07-10 00:00:00+00:00,NaN,NaN,5.6,8.9,343.0,29.2,13.2,0.3,0.3,0.0,0.0,NaN,NaN,24.10,5.414980,0.080071,73.500002,402.300001,625.500002
104129,2026-07-10 01:00:00+00:00,NaN,NaN,5.6,8.9,343.0,29.2,13.2,0.1,0.1,0.0,0.0,NaN,NaN,23.55,7.742093,0.052064,68.300002,402.400001,623.800002
104130,2026-07-10 02:00:00+00:00,NaN,NaN,5.6,8.9,343.0,29.2,13.2,0.0,0.0,0.0,0.0,NaN,NaN,23.30,7.841888,0.051383,63.000002,402.400001,622.000002


In [18]:
# Flood Action Stage: 5 ft
# Minor Flood Stage: 10 ft
# Moderate Flood Stage: 12 ft
# Major Flood Stage: 14 ft
FLOOD_ACTION_STAGE = 5.0
MINOR_FLOOD_STAGE = 10.0
MODERATE_FLOOD_STAGE = 12.0
MAJOR_FLOOD_STAGE = 14.0

In [ ]:
#df.hist(figsize=(10, 6))
# get the instances where Gage Height is > 5
df = df.dropna(subset=['gage_height_ft'])
df_flood = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE]
df_minor_flood = df[df['gage_height_ft'] > MINOR_FLOOD_STAGE]
df_moderate_flood = df[df['gage_height_ft'] > MODERATE_FLOOD_STAGE]
df_major_flood = df[df['gage_height_ft'] > MAJOR_FLOOD_STAGE]

# print the lengths of each of the dataframes
print(f"Total records: {len(df)}")
print(f"Flood Action Stage records: {len(df_flood)}")
print(f"Minor flood records: {len(df_minor_flood)}")
print(f"Moderate flood records: {len(df_moderate_flood)}")
print(f"Major flood records: {len(df_major_flood)}")

Total records: 97742
Flood Action Stage records: 3030
Minor flood records: 0
Moderate flood records: 0
Major flood records: 0


In [ ]:
# df should have a datetime column and a binary flag column for action stage
# adjust column names to match your data

# Name the 'Unnamed: 0' column as 'datetime' and convert it to datetime type
df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

df = df.sort_values('datetime').reset_index(drop=True)

# how many hours of gap counts as "the storm ended" (tune this to your data's
# sampling frequency, e.g. 6-12h for hourly gauge data, 24-48h for daily)
GAP_HOURS = 12

# isolate just the flagged (action-stage) rows
flood_rows = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE].copy()

# time since previous flagged reading, saved in column 'gap'
flood_rows['gap'] = flood_rows['datetime'].diff()

# start a new event whenever the gap exceeds threshold (or it's the first row)
flood_rows['new_event'] = (
    flood_rows['gap'].isna() | (flood_rows['gap'] > pd.Timedelta(hours=GAP_HOURS))
)
flood_rows['event_id'] = flood_rows['new_event'].cumsum()

# summarize each event
events = flood_rows.groupby('event_id').agg(
    start=('datetime', 'min'),
    end=('datetime', 'max'),
    n_readings=('datetime', 'count'),
    peak_gage_height=('gage_height_ft', 'max')  # adjust column name as needed
).reset_index(drop=True)

events['duration_hours'] = (events['end'] - events['start']).dt.total_seconds() / 3600

print(f"Total flagged readings: {len(flood_rows)}")
print(f"Independent storm events: {len(events)}")
print(events)


Total flagged readings: 3030
Independent storm events: 3
                      start                       end  n_readings  \
0 2025-07-19 23:50:00+00:00 2025-07-20 01:35:00+00:00          22   
1 2026-02-21 15:35:00+00:00 2026-02-24 23:50:00+00:00         964   
2 2026-05-24 18:55:00+00:00 2026-05-31 21:10:00+00:00        2044   

   peak_gage_height  duration_hours  
0              5.12            1.75  
1              5.88           80.25  
2              6.46          170.25  
count                         3029
mean     0 days 02:30:10.498514361
std      4 days 06:03:44.038531146
min                0 days 00:05:00
25%                0 days 00:05:00
50%                0 days 00:05:00
75%                0 days 00:05:00
max              216 days 14:00:00
Name: gap, dtype: object
gap
216 days 14:00:00       1
88 days 19:05:00        1
0 days 00:05:00      3027
Name: count, dtype: int64
